# Echogram Clustering

The dimension analysis in notebook 06 showed that echogram files vary considerably in
width (pings) and height (range samples), with what looks like natural groupings in the
scatter plot. This notebook applies clustering algorithms to formalise those groups,
enriches the feature set with patch-level quality metrics, and examines what
distinguishes each cluster — both physically and in terms of training-pipeline behaviour.

Features are derived from two sources: the cached NPZ patch files (dimensions and
pixel statistics) and the per-patch quality metrics CSV (bottom coverage, dynamic
range, frequency coherence, etc.). These are aggregated per recording to build a
rich feature vector for each of the 88 echogram files.

In [ ]:
import sys
sys.path.insert(0, '../../../..')

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from itertools import permutations

sns.set_style('whitegrid')

REPO_ROOT = Path('../../../..').resolve()
DATASETS = REPO_ROOT / 'Datasets'
PATCH_CACHE = DATASETS / 'patch_cache'
QUALITY_CSV = DATASETS / 'patch_quality_metrics.csv'
PATCH_WIDTH = 256
STRIDE = 64

## 1. Feature collection

We aggregate features at the recording level from two sources: the NPZ patch files
(patch count, height, pixel statistics from the three channels) and the quality
metrics CSV (bottom coverage, dynamic range, boundary contrast, frequency coherence).

In [ ]:
# --- NPZ-derived features per recording ---
npz_files = sorted(PATCH_CACHE.glob('*.npz'))

# Group by recording name (stem before hash suffix)
file_groups = defaultdict(list)
for npz_path in npz_files:
    stem = npz_path.stem.rsplit('_', 1)[0]
    file_groups[stem].append(npz_path)

records = []
for file_stem, paths in sorted(file_groups.items()):
    all_patches = []
    all_masks = []
    for p in paths:
        with np.load(p) as data:
            all_patches.append(data['patches'])  # (N, 3, H, W)
            all_masks.append(data['masks'])       # (N, H, W)

    patches = np.concatenate(all_patches, axis=0)
    masks = np.concatenate(all_masks, axis=0)
    n_patches_total = patches.shape[0]
    patch_height = patches.shape[2]

    # Reconstruct approximate echogram width from patch count
    # n_patches = (width - PATCH_WIDTH) // STRIDE + 1  =>  width ≈ (n_patches - 1) * STRIDE + PATCH_WIDTH
    approx_width = (n_patches_total - 1) * STRIDE + PATCH_WIDTH

    # Pixel statistics from channel 0 (38 kHz normalised Sv)
    ch0 = patches[:, 0]  # (N, H, W)
    ch1 = patches[:, 1]  # (N, H, W) — 200 kHz
    ch2 = patches[:, 2]  # (N, H, W) — freq difference

    records.append({
        'file': file_stem,
        'n_patches': n_patches_total,
        'n_range_samples': patch_height,
        'approx_n_pings': approx_width,
        'ch0_mean': float(np.mean(ch0)),
        'ch0_std': float(np.std(ch0)),
        'ch0_p5': float(np.percentile(ch0, 5)),
        'ch0_p95': float(np.percentile(ch0, 95)),
        'ch1_mean': float(np.mean(ch1)),
        'ch1_std': float(np.std(ch1)),
        'ch2_mean': float(np.mean(ch2)),
        'ch2_std': float(np.std(ch2)),
        'mask_fraction': float(np.mean(masks > 0)),
    })

df_npz = pd.DataFrame(records)
print(f'{len(df_npz)} recordings from {len(npz_files)} NPZ files')

In [ ]:
# --- Quality-metrics CSV: aggregate per recording ---
qm = pd.read_csv(QUALITY_CSV)

qm_agg = qm.groupby('source_zarr').agg(
    qm_n_patches=('patch_idx', 'count'),
    zero_frac_mean=('zero_frac', 'mean'),
    zero_frac_std=('zero_frac', 'std'),
    dynamic_range_mean=('dynamic_range', 'mean'),
    dynamic_range_std=('dynamic_range', 'std'),
    water_col_mean=('water_col_mean', 'mean'),
    water_col_std_mean=('water_col_std', 'mean'),
    bottom_coverage_mean=('bottom_coverage', 'mean'),
    bottom_frac_mean=('bottom_frac', 'mean'),
    bottom_frac_std=('bottom_frac', 'std'),
    bottom_smoothness_mean=('bottom_smoothness', 'mean'),
    bottom_depth_range_mean=('bottom_depth_range', 'mean'),
    boundary_contrast_mean=('boundary_contrast', 'mean'),
    freq_coherence_mean=('freq_coherence', 'mean'),
    freq_coherence_std=('freq_coherence', 'std'),
).reset_index().rename(columns={'source_zarr': 'file'})

print(f'{len(qm_agg)} recordings in quality CSV')

In [ ]:
# Merge NPZ and quality metrics
df = df_npz.merge(qm_agg, on='file', how='left')

# Derived features
df['aspect_ratio'] = df['approx_n_pings'] / df['n_range_samples']
df['total_pixels_M'] = (df['approx_n_pings'] * df['n_range_samples']) / 1e6
df['resize_factor'] = 512 / df['n_range_samples']
df['ch0_dynamic_range'] = df['ch0_p95'] - df['ch0_p5']

print(f'{len(df)} recordings, {len(df.columns)} columns')
df[['file', 'n_patches', 'n_range_samples', 'approx_n_pings',
    'ch0_mean', 'dynamic_range_mean', 'freq_coherence_mean']].head(10)

## 2. Feature exploration

A correlation matrix and pair plot show which features carry independent information
and where natural groupings might already be visible.

In [ ]:
# Select features for clustering
feature_cols = [
    'approx_n_pings', 'n_range_samples', 'n_patches',
    'ch0_mean', 'ch0_std', 'ch0_dynamic_range',
    'ch1_mean', 'ch1_std', 'ch2_mean', 'ch2_std',
    'mask_fraction',
    'dynamic_range_mean', 'water_col_mean', 'water_col_std_mean',
    'bottom_coverage_mean', 'bottom_frac_mean',
    'bottom_smoothness_mean', 'boundary_contrast_mean',
    'freq_coherence_mean',
]

# Keep only features with enough non-NaN values
feature_cols = [c for c in feature_cols if c in df.columns and df[c].notna().sum() > len(df) * 0.5]

df_feat = df[feature_cols].copy()
for col in feature_cols:
    df_feat[col] = df_feat[col].fillna(df_feat[col].median())

print(f'{len(feature_cols)} features for clustering:')
for c in feature_cols:
    print(f'  {c}: mean={df_feat[c].mean():.4f}, std={df_feat[c].std():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df_feat.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.7},
            annot_kws={'size': 7})
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
pair_cols = ['approx_n_pings', 'n_range_samples', 'ch0_mean',
             'dynamic_range_mean', 'freq_coherence_mean']
pair_cols = [c for c in pair_cols if c in feature_cols]

g = sns.pairplot(df_feat[pair_cols], diag_kind='kde',
                 plot_kws={'alpha': 0.6, 's': 30})
g.figure.suptitle('Pair plot of key features', y=1.02)
plt.show()

## 3. Choosing k

The elbow plot and silhouette scores guide the choice of cluster count. The visual
impression from notebook 06 suggested around 3, but we test k from 2 to 8.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df_feat)

k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(list(k_range), inertias, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.set_title('Elbow plot')
ax.set_xticks(list(k_range))

ax = axes[1]
ax.plot(list(k_range), silhouettes, 'o-', color='darkorange', linewidth=2)
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette score vs k')
ax.set_xticks(list(k_range))

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'Best silhouette score at k={best_k} ({max(silhouettes):.3f})')

## 4. KMeans clustering

We fit KMeans with k=3 as the baseline (from visual inspection) and also show k=2
and k=4 for comparison. The scatter plots use the same axes as notebook 06's final
plot for easy visual comparison.

In [ ]:
cluster_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, k in zip(axes, [2, 3, 4]):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)

    for c in range(k):
        m = labels == c
        ax.scatter(df.loc[m, 'approx_n_pings'], df.loc[m, 'n_range_samples'],
                   s=60, alpha=0.7, label=f'Cluster {c} (n={m.sum()})',
                   color=cluster_colors[c], edgecolors='k', linewidths=0.4)

    ax.set_xlabel('Number of pings (width)')
    ax.set_ylabel('Number of range samples (height)')
    ax.set_title(f'KMeans k={k}  (silhouette={sil:.3f})')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Use the best-silhouette k going forward
K = best_k
km_final = KMeans(n_clusters=K, n_init=20, random_state=42)
df['cluster'] = km_final.fit_predict(X)

print(f'Assigned {K} clusters:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Silhouette plot
sil_vals = silhouette_samples(X, df['cluster'].values)
avg_sil = silhouette_score(X, df['cluster'].values)

fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10

for c in range(K):
    c_vals = np.sort(sil_vals[df['cluster'] == c])
    size = len(c_vals)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_vals,
                     alpha=0.7, color=cluster_colors[c], label=f'Cluster {c}')
    ax.text(-0.05, y_lower + 0.5 * size, str(c), fontsize=12, fontweight='bold')
    y_lower = y_upper + 10

ax.axvline(avg_sil, color='k', ls='--', lw=1.5, label=f'Average = {avg_sil:.3f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Files (sorted within cluster)')
ax.set_title(f'Silhouette plot — KMeans k={K}')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. DBSCAN

A density-based alternative that finds clusters without specifying k. The k-distance
plot helps choose the eps parameter, then we try a few values.

In [ ]:
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X)
distances, _ = nn.kneighbors(X)
k_dist = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_dist, color='steelblue', linewidth=2)
ax.set_xlabel('Files (sorted by 5-NN distance)')
ax.set_ylabel('5th nearest-neighbour distance')
ax.set_title('k-distance plot for DBSCAN eps selection')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, eps in zip(axes, [2.0, 3.0, 4.0]):
    db = DBSCAN(eps=eps, min_samples=3)
    labels = db.fit_predict(X)
    n_clusters = len(set(labels) - {-1})
    n_noise = (labels == -1).sum()

    for c in sorted(set(labels)):
        m = labels == c
        if c == -1:
            ax.scatter(df.loc[m, 'approx_n_pings'], df.loc[m, 'n_range_samples'],
                       s=40, alpha=0.5, marker='x', color='gray',
                       label=f'Noise (n={m.sum()})')
        else:
            ax.scatter(df.loc[m, 'approx_n_pings'], df.loc[m, 'n_range_samples'],
                       s=60, alpha=0.7, label=f'Cluster {c} (n={m.sum()})',
                       color=cluster_colors[c % len(cluster_colors)],
                       edgecolors='k', linewidths=0.4)

    ax.set_xlabel('Number of pings (width)')
    ax.set_ylabel('Number of range samples (height)')
    ax.set_title(f'DBSCAN eps={eps}  ({n_clusters} clusters, {n_noise} noise)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Hierarchical clustering

Ward linkage builds a tree of cluster merges. The dendrogram shows how files group
together and where the natural cut points are.

In [ ]:
Z = linkage(X, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=8,
           color_threshold=Z[-(K-1), 2])
ax.set_xlabel('File index (or cluster size)')
ax.set_ylabel('Ward distance')
ax.set_title('Hierarchical clustering dendrogram (Ward linkage)')
ax.axhline(Z[-(K-1), 2], color='gray', ls=':', lw=1, alpha=0.7,
           label=f'Cut for {K} clusters')
ax.legend()
plt.tight_layout()
plt.show()

# Compare hierarchical assignments to KMeans
df['cluster_hier'] = fcluster(Z, t=K, criterion='maxclust') - 1

best_agree = 0
for perm in permutations(range(K)):
    remapped = df['cluster_hier'].map(dict(enumerate(perm)))
    agree = (df['cluster'] == remapped).mean()
    best_agree = max(best_agree, agree)
print(f'KMeans vs hierarchical agreement (best label permutation): {best_agree:.1%}')

## 7. Per-cluster statistics

Summary statistics for each cluster across all features, with bar charts and box
plots comparing the clusters.

In [ ]:
stats_cols = [
    'approx_n_pings', 'n_range_samples', 'n_patches',
    'aspect_ratio', 'total_pixels_M', 'resize_factor',
    'ch0_mean', 'ch0_std', 'ch0_dynamic_range',
    'mask_fraction',
    'dynamic_range_mean', 'water_col_mean',
    'bottom_coverage_mean', 'bottom_frac_mean',
    'boundary_contrast_mean', 'freq_coherence_mean',
]
stats_cols = [c for c in stats_cols if c in df.columns]

summary = df.groupby('cluster')[stats_cols].agg(['mean', 'std']).round(3)
print(f'Per-cluster statistics (k={K}):\n')
summary

In [ ]:
bar_cols = ['approx_n_pings', 'n_range_samples', 'n_patches',
            'total_pixels_M', 'ch0_mean', 'freq_coherence_mean']
bar_cols = [c for c in bar_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, bar_cols):
    means = df.groupby('cluster')[col].mean()
    stds = df.groupby('cluster')[col].std()
    bars = ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
                  color=[cluster_colors[i] for i in means.index],
                  edgecolor='white', alpha=0.85)
    ax.set_xlabel('Cluster')
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.set_xticks(means.index)

for ax in axes[len(bar_cols):]:
    ax.set_visible(False)

plt.suptitle(f'Cluster comparison (k={K})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
box_cols = ['approx_n_pings', 'n_range_samples', 'ch0_mean', 'ch0_std',
            'dynamic_range_mean', 'freq_coherence_mean']
box_cols = [c for c in box_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, box_cols):
    data_per_cluster = [df.loc[df['cluster'] == c, col].values for c in range(K)]
    bp = ax.boxplot(data_per_cluster, patch_artist=True,
                    tick_labels=[str(c) for c in range(K)])
    for patch, color in zip(bp['boxes'], cluster_colors[:K]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xlabel('Cluster')
    ax.set_ylabel(col)
    ax.set_title(col)

for ax in axes[len(box_cols):]:
    ax.set_visible(False)

plt.suptitle(f'Feature distributions per cluster (k={K})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Cluster characterisation

Parallel coordinates and a radar chart show each cluster's profile across features
(standardised to z-scores).

In [ ]:
plot_cols = ['approx_n_pings', 'n_range_samples', 'ch0_mean', 'ch0_std',
             'mask_fraction', 'dynamic_range_mean', 'bottom_frac_mean',
             'boundary_contrast_mean', 'freq_coherence_mean']
plot_cols = [c for c in plot_cols if c in feature_cols]

# Standardise for display
df_plot = df[plot_cols].copy()
for col in plot_cols:
    col_std = df_plot[col].std()
    if col_std > 0:
        df_plot[col] = (df_plot[col] - df_plot[col].mean()) / col_std
    else:
        df_plot[col] = 0.0
df_plot['cluster'] = df['cluster'].values

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(plot_cols))

for c in range(K):
    subset = df_plot[df_plot['cluster'] == c]
    for _, row in subset.iterrows():
        ax.plot(x, row[plot_cols].values, color=cluster_colors[c],
                alpha=0.15, linewidth=1)
    means = subset[plot_cols].mean()
    ax.plot(x, means.values, color=cluster_colors[c], linewidth=3, alpha=0.9,
            label=f'Cluster {c} mean (n={len(subset)})', marker='o', markersize=5)

ax.set_xticks(x)
ax.set_xticklabels(plot_cols, rotation=30, ha='right')
ax.set_ylabel('Standardised value (z-score)')
ax.set_title('Parallel coordinates — cluster profiles')
ax.legend(loc='upper right')
ax.axhline(0, color='gray', ls=':', lw=1)
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart
radar_cols = plot_cols.copy()
n_vars = len(radar_cols)
angles = np.linspace(0, 2 * np.pi, n_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for c in range(K):
    vals = df_plot[df_plot['cluster'] == c][radar_cols].mean().values.tolist()
    vals += vals[:1]
    ax.plot(angles, vals, color=cluster_colors[c], linewidth=2, label=f'Cluster {c}')
    ax.fill(angles, vals, color=cluster_colors[c], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_cols, fontsize=8)
ax.set_title('Cluster profiles (standardised means)', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
# Scatter with cluster labels and size encoding total pixels
fig, ax = plt.subplots(figsize=(10, 7))

for c in range(K):
    m = df['cluster'] == c
    sizes = df.loc[m, 'total_pixels_M'] / df['total_pixels_M'].max() * 300
    ax.scatter(df.loc[m, 'approx_n_pings'], df.loc[m, 'n_range_samples'],
               s=sizes, alpha=0.7, label=f'Cluster {c} (n={m.sum()})',
               color=cluster_colors[c], edgecolors='k', linewidths=0.5)

ax.axvline(PATCH_WIDTH, color='gray', ls=':', lw=1, alpha=0.5)
ax.text(PATCH_WIDTH + 30, ax.get_ylim()[0] + 50,
        f'patch width = {PATCH_WIDTH}', fontsize=8, color='gray')

ax.set_xlabel('Approx. number of pings (width)')
ax.set_ylabel('Number of range samples (height)')
ax.set_title(f'Echogram clusters (k={K}) — marker size proportional to total pixels')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Training implications

How the clusters relate to what the U-Net sees during training: the number of
patches per cluster, the vertical resize factor, and overall balance.

In [ ]:
train_stats = df.groupby('cluster').agg(
    n_files=('file', 'count'),
    total_patches=('n_patches', 'sum'),
    avg_patches_per_file=('n_patches', 'mean'),
    avg_resize_factor=('resize_factor', 'mean'),
    avg_pings=('approx_n_pings', 'mean'),
    avg_range_samples=('n_range_samples', 'mean'),
    avg_ch0_mean=('ch0_mean', 'mean'),
    avg_freq_coherence=('freq_coherence_mean', 'mean'),
).round(2)

train_stats['patch_fraction_pct'] = (
    train_stats['total_patches'] / train_stats['total_patches'].sum() * 100
).round(1)
train_stats['file_fraction_pct'] = (
    train_stats['n_files'] / train_stats['n_files'].sum() * 100
).round(1)

print('Training pipeline impact per cluster:\n')
train_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Files per cluster
ax = axes[0]
bars = ax.bar(train_stats.index, train_stats['n_files'],
              color=[cluster_colors[i] for i in train_stats.index],
              edgecolor='white')
for bar, pct in zip(bars, train_stats['file_fraction_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.0f}%', ha='center', fontsize=10)
ax.set_xlabel('Cluster')
ax.set_ylabel('Number of files')
ax.set_title('Files per cluster')

# Patches per cluster
ax = axes[1]
bars = ax.bar(train_stats.index, train_stats['total_patches'],
              color=[cluster_colors[i] for i in train_stats.index],
              edgecolor='white')
for bar, pct in zip(bars, train_stats['patch_fraction_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{pct:.0f}%', ha='center', fontsize=10)
ax.set_xlabel('Cluster')
ax.set_ylabel('Total patches')
ax.set_title('Patches per cluster (stride=64)')

# Resize factor distribution
ax = axes[2]
data_per_cluster = [df.loc[df['cluster'] == c, 'resize_factor'].values
                    for c in range(K)]
bp = ax.boxplot(data_per_cluster, patch_artist=True,
                tick_labels=[str(c) for c in range(K)])
for patch, color in zip(bp['boxes'], cluster_colors[:K]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel('Cluster')
ax.set_ylabel('Vertical resize factor (to 512 px)')
ax.set_title('Resize factor per cluster')

plt.suptitle('Training pipeline balance across clusters', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()